## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Testing to ensure I connected to an upgraded GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Sep 13 23:58:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
# pip install numpy==2.1.2 --force-reinstall
#CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q
# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q
# --------------------------New Code ---------------------------------------------
# Packages I had to install to ge tthis work

!apt-get install cmake build-essential -y
!pip install --upgrade pip
!pip install --upgrade setuptools wheel


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.6 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [1]:
!pip install llama-cpp-python --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 157.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4500213 sha256=de05f3a6e6b1e5b65b671dfb67f41974c8b5fd02687ef5484edf8be4a512413e
  Stored in directory: /tmp/pip-ephem-wheel-cache-iveko342/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [2]:
# Access to Google Drive so I could download the model once and use it from my drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import time
my_start_time = time.time()
# my_start_time

In [4]:
from llama_cpp import Llama

In [5]:
# invoking an instance of the model
model_path = "/content/drive/MyDrive/models/mistral-7b-instruct-v0.2.Q6_K.gguf"

llm = Llama(
    model_path=model_path,
    n_threads=16,   # adjust according to your GPU/CPU
    n_batch=256,     # VRAM
    n_gpu_layers=32,
    n_ctx=4096     # context window
)

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/drive/MyDrive/models/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader:

In [6]:
# Model Test
prompt = "Write a short story about a friendly robot."
output = llm(prompt, max_tokens=200)
# print(output)
print("Metadata:", output.get("choices"))


llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =     939.42 ms /    10 tokens (   93.94 ms per token,    10.64 tokens per second)
llama_perf_context_print:        eval time =   39187.22 ms /   199 runs   (  196.92 ms per token,     5.08 tokens per second)
llama_perf_context_print:       total time =   40257.06 ms /   209 tokens
llama_perf_context_print:    graphs reused =        192


Metadata: [{'text': '\n\nIn the heart of the bustling city of Neo-Tokyo, there was a robot named Kibo. Kibo was not like any other robot in the city. He was designed to be a companion, a friend, to the lonely elderly population. With his warm, expressive eyes and gentle, soothing voice, he brought joy and companionship to many lonely lives.\n\nEvery day, Kibo would make his rounds, visiting the elderly in their homes. He would help them with their daily chores, prepare them meals, and even engage them in conversation. Kibo was programmed with a vast knowledge of human history, culture, and trivia, and he loved nothing more than sharing stories with his friends.\n\nOne sunny afternoon, as Kibo was visiting Ms. Takahashi, an elderly woman who lived alone in a quiet apartment, he noticed that she looked particularly sad. She sat quietly in her chair,', 'index': 0, 'logprobs': None, 'finish_reason': 'length'}]


In [7]:
!pip install langchain-community
!pip install chromadb
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.5 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 194.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 83.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 187.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 14

In [8]:
# Importing needed modules
import pandas as pd
import numpy as np
import huggingface_hub
import packaging
import langchain
import langchain_community
import langchain_text_splitters
import chromadb
import sentence_transformers
import pymupdf
import tiktoken

In [9]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

In this section each questions will be asked 3 times.

1.   As stated with no modificaitons
2.   With prompt engineering
3.   RAG with prompt engineering

After which we will evaluate the 3 levels of responses


#### Response

In [10]:
def response(query,max_tokens=256,temperature=0.8,top_p=0.9,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: 1 prefix-match hit, remaining 15 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    2691.40 ms /    15 tokens (  179.43 ms per token,     5.57 tokens per second)
llama_perf_context_print:        eval time =   51339.72 ms /   255 runs   (  201.33 ms per token,     4.97 tokens per second)
llama_perf_context_print:       total time =   54215.83 ms /   270 tokens
llama_perf_context_print:    graphs reused =        246


"\n\nSepsis is a life-threatening condition that arises from a dysregulated host response to infection. Early recognition and prompt intervention are crucial to improving outcomes. The following are the key steps in managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis, which can include fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis. Use validated scoring systems such as the Sequential Organ Failure Assessment (SOFA) score or the Quick Sequential Organ Failure Assessment (qSOFA) score to help identify patients at risk for sepsis.\n2. Resuscitation: Provide adequate fluid resuscitation to maintain adequate tissue perfusion. Use crystalloids initially and consider colloids or blood products if the patient's response is inadequate. Maintain a mean arterial pressure (MAP) of at least 65 mmHg and a central venous pressure (CVP) of 8-12 mmHg in sedated, mechanically ven

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input)

Llama.generate: 2 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    2279.31 ms /    32 tokens (   71.23 ms per token,    14.04 tokens per second)
llama_perf_context_print:        eval time =   50609.54 ms /   255 runs   (  198.47 ms per token,     5.04 tokens per second)
llama_perf_context_print:       total time =   53072.15 ms /   287 tokens
llama_perf_context_print:    graphs reused =        246


'\n\nAppendicitis is a common inflammatory condition that affects the appendix, a small tube-shaped structure that extends from the cecum, the first part of the large intestine. The symptoms of appendicitis can vary, but they typically include:\n\n1. Abdominal pain: The pain is usually located in the lower right side of the abdomen and may be constant or come and go. It may begin as a mild discomfort that progresses to sharp, localized pain.\n2. Loss of appetite: A loss of appetite or feeling sick to your stomach may be an early symptom of appendicitis.\n3. Nausea and vomiting: Nausea and vomiting are common symptoms of appendicitis, and they may occur as the inflammation in the appendix presses on the stomach.\n4. Fever: A fever is often present in appendicitis, especially if the inflammation has been ongoing for some time.\n5. Diarrhea or constipation: Depending on the location and severity of the inflammation, you may experience diarrhea or constipation.\n\nAppendicitis cannot be cu

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [13]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input)

Llama.generate: 4 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    2439.26 ms /    34 tokens (   71.74 ms per token,    13.94 tokens per second)
llama_perf_context_print:        eval time =   50502.46 ms /   255 runs   (  198.05 ms per token,     5.05 tokens per second)
llama_perf_context_print:       total time =   53124.60 ms /   289 tokens
llama_perf_context_print:    graphs reused =        246


'\n\nSudden patchy hair loss, also known as alopecia areata, is a common hair disorder that affects both men and women. It is characterized by the sudden appearance of small, round bald patches on the scalp, which can range from a few hairs to a large area. The good news is that there are several treatments and solutions for addressing this condition.\n\n1. Topical corticosteroids: Topical corticosteroids are the most commonly used treatment for alopecia areata. They are applied directly to the affected area in the form of creams, gels, or solutions. Topical corticosteroids work by reducing inflammation and suppressing the immune system, which helps to promote hair regrowth.\n2. Injections of corticosteroids: Injections of corticosteroids, such as triamcinolone acetonide or methylprednisolone, can be used to treat larger bald patches. The injections are given into the affected area every 4 to 6 weeks, and they can help to stimulate hair regrowth.\n3. Minoxidil: Minoxidil is a medicatio

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [14]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input)

Llama.generate: 2 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    2019.58 ms /    28 tokens (   72.13 ms per token,    13.86 tokens per second)
llama_perf_context_print:        eval time =   50360.99 ms /   255 runs   (  197.49 ms per token,     5.06 tokens per second)
llama_perf_context_print:       total time =   52564.60 ms /   283 tokens
llama_perf_context_print:    graphs reused =        246


"\n\nA person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, is said to have a brain injury. Brain injuries can range from mild, such as a concussion, to severe, such as a traumatic brain injury (TBI). Treatment for brain injuries depends on the severity and location of the injury, as well as the individual's age, overall health, and other factors.\n\nHere are some common treatments for brain injuries:\n\n1. Medical care: The first priority is to ensure the person's airway is clear, they are breathing and have adequate circulation. If the injury is severe, they may require emergency surgery or other medical interventions to prevent further damage.\n2. Rehabilitation: Rehabilitation is an essential component of treatment for brain injuries. Rehabilitation may include physical therapy, occupational therapy, speech therapy, and other types of therapy to help the person regain lost functions and improve their overall q

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [15]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery"
response(user_input)

Llama.generate: 2 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    2469.22 ms /    34 tokens (   72.62 ms per token,    13.77 tokens per second)
llama_perf_context_print:        eval time =   50410.84 ms /   255 runs   (  197.69 ms per token,     5.06 tokens per second)
llama_perf_context_print:       total time =   53063.89 ms /   289 tokens
llama_perf_context_print:    graphs reused =        246


'?\n\nA fractured leg is a serious injury that requires prompt medical attention. Here are some necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Call for emergency medical help: If the injury is severe, call for emergency medical help. Do not try to move the person unless it is necessary to keep them safe or prevent further injury.\n2. Improvise a splint: If you are alone with the injured person, try to immobilize the leg by creating a splint using available materials such as branches, sticks, or clothing. Be sure to secure the splint above and below the injury to prevent any further movement.\n3. Keep the injured person warm and comfortable: Keep the injured person warm and comfortable by covering them with blankets or clothing. Provide them with water and food if they are able to consume it.\n4. Do not attempt to realign the bone: Do not try to realign the bone or put it back into place yourself. This could cause further d

## Question Answering using LLM with Prompt Engineering

In [16]:
# function to generate, process, and return the response from the LLM
def generate_llama_response(user_prompt):

    # System message
    system_message = """
    [INST]<<SYS>> Respond to the user question based on the user prompt<</SYS>>[/INST]
    """

    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    # Generate a response from the LLaMA model
    response = llm(
        prompt=prompt,
        max_tokens=1024,
        temperature=0.01,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"]
    return response_text

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [17]:
user_input = "I am the manager of a critical care unit and I need a detailed protocal for managing and preventing a sepsis case in my unit? Please provide step-by-step instructions for managing sepsis in a critical care unti. Tone shoould be professional in bullet points"
generate_llama_response(user_input)

Llama.generate: 1 prefix-match hit, remaining 89 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    6267.11 ms /    89 tokens (   70.42 ms per token,    14.20 tokens per second)
llama_perf_context_print:        eval time =  206033.02 ms /  1015 runs   (  202.99 ms per token,     4.93 tokens per second)
llama_perf_context_print:       total time =  214226.13 ms /  1104 tokens
llama_perf_context_print:    graphs reused =        982


"1. **Suspicion of Sepsis:** Recognize the signs and symptoms of sepsis, which may include: fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, lactic acidosis, and decreased urine output.\n    2. **Early Recognition:** Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to identify patients at risk for developing sepsis or worsening of their current condition. A SOFA score ≥ 2 indicates a higher likelihood of sepsis.\n    3. **Immediate Action:** Initiate resuscitation measures as per Advanced Cardiac Life Support (ACLS) guidelines if the patient shows signs of cardiovascular instability, such as hypotension or low urine output.\n        1. Maintain adequate oxygenation and ventilation using non-invasive or invasive mechanical ventilation if necessary.\n        2. Administer intravenous fluids to maintain an adequate circulating volume (goal: mean arterial pressure ≥65 mmHg).\n    4. **Antibiotics:** Start broad-spectrum

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [18]:
user_input = "I am a doctor in a new clinic and I have a patient that may have appendicitis. What are the  common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_llama_response(user_input)

Llama.generate: 3 prefix-match hit, remaining 82 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    5758.92 ms /    82 tokens (   70.23 ms per token,    14.24 tokens per second)
llama_perf_context_print:        eval time =   67914.72 ms /   340 runs   (  199.75 ms per token,     5.01 tokens per second)
llama_perf_context_print:       total time =   74126.05 ms /   422 tokens
llama_perf_context_print:    graphs reused =        328


"\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms for appendicitis:\n\n1. Sudden pain in the lower right belly area that may begin as mild and then becomes sharp and severe over time.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Fever, which can be low-grade at first but rises as the condition worsens.\n4. Vomiting, especially in later stages.\n5. Constipation or diarrhea.\n6. Abdominal swelling and rigidity.\n7. Pain that worsens when you walk, cough, sneeze, or make other jarring movements.\n8. Inability to pass gas or have a bowel movement.\n9. Feeling restless or unable to find a comfortable position due to pain.\n\nAppendicitis cannot be cured with medicine alone as the inflammation and swelling can obstruct the appendix, leading to rupture if left untreated. If diagnosed early, antibiotics may help prevent complicat

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [19]:
user_input = "I am a doctor and my patient has sudden patchy hair loss seen as localized bald spots. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_llama_response(user_input)

Llama.generate: 5 prefix-match hit, remaining 83 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    7123.41 ms /    83 tokens (   85.82 ms per token,    11.65 tokens per second)
llama_perf_context_print:        eval time =  114390.48 ms /   573 runs   (  199.63 ms per token,     5.01 tokens per second)
llama_perf_context_print:       total time =  122361.72 ms /   656 tokens
llama_perf_context_print:    graphs reused =        554


'1. Possible Causes:\n   a) Alopecia Areata: An autoimmune disorder that causes hair loss in round patches on the scalp, face, or other areas of the body. It often starts with one or more small, smooth patches but may advance to complete baldness (alopecia totalis), or involve the entire body (alopecia universalis).\n   b) Traction Alopecia: Hair loss caused by pulling on the hair too tightly, such as from braids, ponytails, or extensions. This condition is more common in individuals with tight curls and can lead to permanent damage if not addressed promptly.\n   c) Tinea Capitis (Ringworm): A fungal infection that affects the scalp and hair follicles, causing circular patches of hair loss with redness, scaling, or inflammation.\n   d) Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, biotin, vitamin D, or protein can lead to patchy hair loss.\n   e) Stress: Emotional stressors such as illness, surgery, or significant life changes may cause temporary hair shedding 

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [20]:
user_input = "I have a family member that sustained a physical injury to brain tissue. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_llama_response(user_input)

Llama.generate: 2 prefix-match hit, remaining 72 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    6268.99 ms /    72 tokens (   87.07 ms per token,    11.49 tokens per second)
llama_perf_context_print:        eval time =   75618.35 ms /   381 runs   (  198.47 ms per token,     5.04 tokens per second)
llama_perf_context_print:       total time =   82415.74 ms /   453 tokens
llama_perf_context_print:    graphs reused =        368


"I'm an assistant and I can provide some general information about treatments for physical injuries to the brain. However, it is important to note that every individual's condition is unique, so specific treatment recommendations should be discussed with a healthcare professional.\n\n    1. Medical Management: Depending on the severity of the injury, initial medical management may include controlling vital signs, managing swelling or bleeding in the brain (hematoma), and addressing any life-threatening conditions.\n\n    2. Rehabilitation Therapies: These therapies can help improve function and independence following a brain injury. They may include physical therapy to address mobility issues, occupational therapy for fine motor skills and daily living tasks, speech therapy for communication difficulties, and cognitive rehabilitation for memory, attention, and problem-solving abilities.\n\n    3. Medications: Depending on the symptoms present, medications may be prescribed to manage co

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [21]:
user_input = "I have just broken my leg on a hiking trip and do not have access to a doctor. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_llama_response(user_input)

Llama.generate: 3 prefix-match hit, remaining 84 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =    7439.00 ms /    84 tokens (   88.56 ms per token,    11.29 tokens per second)
llama_perf_context_print:        eval time =   87786.88 ms /   440 runs   (  199.52 ms per token,     5.01 tokens per second)
llama_perf_context_print:       total time =   95824.91 ms /   524 tokens
llama_perf_context_print:    graphs reused =        425


"1. Assess the severity of the injury: If you suspect that the fracture is severe or open (compound), try to keep the injured area as still as possible and avoid moving it. Do not attempt to realign the bone yourself, as this could cause further damage.\n    2. Control bleeding: Apply direct pressure with a clean cloth to control any bleeding from the injury site. Elevate the leg if possible to reduce swelling and help slow down the flow of blood.\n    3. Immobilize the injured limb: Use a makeshift splint, such as a stick or branch, to immobilize the fractured leg. Make sure it is securely in place but does not restrict circulation. You can also use clothing items like bandanas or shirts to create a sling for support and stability.\n    4. Pain management: Use over-the-counter pain medications if available, such as acetaminophen or ibuprofen, to help manage the pain. If the pain is severe, try to find ways to make yourself more comfortable while waiting for medical assistance.\n    5.

## Data Preparation for RAG

### Loading the Data

In [22]:
doc_pdf_path = "/content/drive/MyDrive/models/med_doc.pdf"

In [23]:
pdf_loader = PyMuPDFLoader(doc_pdf_path)

In [24]:
med = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [25]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(med[i].page_content,end="\n")

Page Number : 1
cmurray4492@gmail.com
E0MHFN73CG
ant for personal use by cmurray4492@g
shing the contents in part or full is liable
Page Number : 2
cmurray4492@gmail.com
E0MHFN73CG
This file is meant for personal use by cmurray4492@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53

In [26]:
med[25].page_content

"Professor J. Wollensak via ONJOPH (Plates 8, 9, 15, 16, 17 [top], and 18); Dr. Brooks McCuen via\nONJOPH (Plate 11 [top]); Jonathan J. Dutton via ONJOPH (Plate 12); University of California at San\nDiego via ONJOPH (Plate 13); Professor H.J. Meyer via ONJOPH (Plate 14); Dr. James Garrity (Plate\n19); World Health Organization via ONJOPH (Plate 20); C. Newman (Plate 21) and B. Biller (Plate 22)\nfrom Atlas of Clinical Endocrinology: Neuroendocrinology and Pituitary Disease ; Thomas Habif, MD\n(Plates 24, 26-30, 32-37, 38 [top], 39-47, 49, 51-55, and 60); Prof. Dr. K.W. Ruprecht and Dr. Med. B.\nKaesmann-Kellner via ONJOPH (Plate 25 [top]); Allen W. Mathies, MD, via the Public Health Image\nLibrary of the Centers for Disease Control and Prevention (CDC) (Plate 31); Dennis D. Juranek, MD, via\nthe Public Health Image Library of the CDC (Plate 38 [bottom]) and via www.doctorfungus.com (Plates\n48 and 50); S. Deitcher from Atlas of Clinical Hematology  (Plate 56); E. Joe and N. Soter from 

#### Checking the number of pages

In [27]:
len(med)

4114

### Data Chunking

In [72]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=90
)

In [73]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [74]:
len(document_chunks)

8973

In [75]:
document_chunks[0].page_content

'cmurray4492@gmail.com\nE0MHFN73CG\nant for personal use by cmurray4492@g\nshing the contents in part or full is liable'

In [78]:
document_chunks[135].page_content

'triamterene, and trimethoprim impair folate metabolism.\nIn the US, many dietary staples (eg, cereals, grain products) are routinely enriched with folate, tending to\nreduce risk of deficiency.\n[Table 4-3. Potential Vitamin-Drug Interactions]\nSymptoms and Signs\nFolate deficiency may cause glossitis, diarrhea, depression, and confusion. Anemia may develop\ninsidiously and, because of compensatory mechanisms, be more severe than symptoms suggest.\nFolate deficiency during pregnancy increases the risk of fetal neural tube defects and perhaps other brain\ndefects (see p. 2992).\nDiagnosis\n• CBC and serum vitamin B12 and folate levels\nCBC may indicate megaloblastic anemia indistinguishable from that of vitamin B12 deficiency. If serum\nfolate is < 3 μg/L or ng/mL (< 7 nmol/L), deficiency is likely. Serum folate reflects folate status unless\nintake has recently increased or decreased. If intake has changed, erythrocyte (RBC) folate level better\nreflects tissue stores. A level of < 14

In [79]:
document_chunks[136].page_content

'vitamin B12 and vitamin B6 levels, renal insufficiency, and genetic factors). A normal methylmalonic acid\n(MMA) level may differentiate folate deficiency from vitamin B12 deficiency because MMA levels rise in\nvitamin B12 deficiency but not in folate deficiency.\nTreatment\n• Supplemental oral folate\nFolate 400 to 1000 μg po once/day replenishes tissues and is usually successful even if deficiency has\nresulted from malabsorption. The normal requirement is 400 μg/day.\n[Table 4-4. Causes of Folate Deficiency]\n(CAUTION: In patients with megaloblastic anemia, vitamin B12 deficiency must be ruled out before\ntreating with folate. If vitamin B12 deficiency is present, folate supplementation can alleviate the\nanemia but does not reverse and may even worsen neurologic deficits.) For pregnant women, the\nrecommended daily allowance (RDA) is 600 μg/day. For women who have had a fetus or infant with a\nneural tube defect, the recommended dose is 1000 to 5000 μg/day.\nNiacin\nNiacin (nicoti

### Embedding

In [35]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipython-input-4198310515.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or dataset

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [36]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [37]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

### Vector Database

In [38]:
out_dir = 'med_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [39]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [40]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipython-input-2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [41]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [42]:
vectorstore.similarity_search("Medical treatment for a broken leg ",k=3)

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/drive/MyDrive/models/med_doc.pdf', 'file_path': '/content/drive/MyDrive/models/med_doc.pdf', 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'page': 3390, 'modDate': 'D:20250831235310Z', 'creator': 'Atop CHM to PDF Converter', 'subject': '', 'keywords': '', 'creationDate': 'D:20120615054440Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2025-08-31T23:53:10+00:00', 'trapped': '', 'total_pages': 4114, 'author': ''}, page_content='(eg, meniscal tears, cartilaginous injuries). Arteriography may be necessary for suspected arterial injuries\n(eg, some popliteal artery injuries). Nerve conduction studies may be indicated for nerve injuries.\nTreatment\n• Treatment of life- or limb-threatening injuries\n• Splinting\n• Definitive treatment (eg, reduction) for certain injuries\n• Rest, ice, compression, and elevation (RICE)\n• Usually immobilizat

### Retriever

In [43]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [44]:
rel_docs = retriever.get_relevant_documents("What are the best ways to treat diabetes?")
rel_docs

/tmp/ipython-input-2322937578.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What are the best ways to treat diabetes?")


[Document(metadata={'source': '/content/drive/MyDrive/models/med_doc.pdf', 'format': 'PDF 1.7', 'keywords': '', 'author': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'subject': '', 'total_pages': 4114, 'creationdate': '2012-06-15T05:44:40+00:00', 'file_path': '/content/drive/MyDrive/models/med_doc.pdf', 'creator': 'Atop CHM to PDF Converter', 'modDate': 'D:20250831235310Z', 'page': 1020, 'creationDate': 'D:20120615054440Z', 'moddate': '2025-08-31T23:53:10+00:00', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'trapped': ''}, page_content='or continuous insulin infusion, is more effective than conventional treatment (1 to 2 insulin injections daily\nwith or without monitoring) for preventing diabetic retinopathy, nephropathy, and neuropathy. However,\nintensive therapy may result in more frequent episodes of hypoglycemia and weight gain and is generally\neffective only in patients who are able and willing to take an active role in their self-care.\

### System and User Prompt Template

In [45]:
qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [46]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [47]:
import textwrap

def generate_rag_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50, wrap_width=80):
    """
    Generates a response using a RAG system and wraps output lines to fit screen width.

    Parameters:
    - user_input: str, the question or query
    - k: int, number of relevant documents to retrieve
    - max_tokens: int, max tokens for LLM
    - temperature: float
    - top_p: float
    - top_k: int
    - wrap_width: int, max line length for wrapping

    Returns:
    - str: wrapped response
    """
    global qna_system_message, qna_user_message_template

    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    # Prepare user message
    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        raw_response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )

        # Extract the model's text
        response = raw_response['choices'][0]['text'].strip()

        # Wrap text to desired width
        wrapped_response = "\n".join(textwrap.wrap(response, width=wrap_width))
    except Exception as e:
        wrapped_response = f"Sorry, I encountered the following error: \n{e}"

    return wrapped_response


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [48]:
user_input = "I am the manager of a critical care unit and I need a detailed protocal for managing and preventing a sepsis case in my unit? Please provide step-by-step instructions for managing sepsis in a critical care unti. Tone shoould be professional in bullet points"
print(generate_rag_response(user_input))

Llama.generate: 1 prefix-match hit, remaining 1943 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  154015.61 ms /  1943 tokens (   79.27 ms per token,    12.62 tokens per second)
llama_perf_context_print:        eval time =   27297.24 ms /   127 runs   (  214.94 ms per token,     4.65 tokens per second)
llama_perf_context_print:       total time =  181388.16 ms /  2070 tokens
llama_perf_context_print:    graphs reused =        122


- Identify and suspect sepsis based on clinical presentation:   * Fever or
hypothermia   * Tachycardia or bradycardia   * Tachypnea or respiratory distress
* Altered mental status   * WBC count > 12,000 cells/μL or < 4,000 cells/μL or >
10% immature forms  - Obtain cultures and initiate broad-spectrum antibiotics:
* Blood, body fluids, and wound sites   * Start


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [49]:
user_input = "I am a doctor in a new clinic and I have a patient that may have appendicitis. What are the  common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 1353 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  106273.94 ms /  1353 tokens (   78.55 ms per token,    12.73 tokens per second)
llama_perf_context_print:        eval time =   12067.67 ms /    58 runs   (  208.06 ms per token,     4.81 tokens per second)
llama_perf_context_print:       total time =  118372.13 ms /  1411 tokens
llama_perf_context_print:    graphs reused =         55


###Answer The common symptoms for appendicitis include abdominal pain, anorexia,
and abdominal tenderness. Appendicitis cannot be cured via medicine and requires
surgical removal, specifically an open or laparoscopic appendectomy.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [50]:
user_input = "I am a doctor and my patient has sudden patchy hair loss seen as localized bald spots. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 1621 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  128887.13 ms /  1621 tokens (   79.51 ms per token,    12.58 tokens per second)
llama_perf_context_print:        eval time =   26379.68 ms /   124 runs   (  212.74 ms per token,     4.70 tokens per second)
llama_perf_context_print:       total time =  155339.86 ms /  1745 tokens
llama_perf_context_print:    graphs reused =        119


###Answer The effective treatments for sudden patchy hair loss, also known as
alopecia areata, include topical, intralesional, or systemic corticosteroids,
topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or
squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of
alopecia areata is believed to be an autoimmune disorder affecting genetically
susceptible people exposed to unclear environmental triggers.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [51]:
user_input = "I have a family member that sustained a physical injury to brain tissue. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 1113 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =   84914.30 ms /  1113 tokens (   76.29 ms per token,    13.11 tokens per second)
llama_perf_context_print:        eval time =   26456.36 ms /   127 runs   (  208.32 ms per token,     4.80 tokens per second)
llama_perf_context_print:       total time =  111446.92 ms /  1240 tokens
llama_perf_context_print:    graphs reused =        122


Your answer: The recommended treatments for a person who has sustained a
physical injury to brain tissue, resulting in temporary or permanent impairment
of brain function, include ensuring a reliable airway and maintaining adequate
ventilation, oxygenation, and blood pressure. Surgery may be needed in patients
with more severe injury to place monitors to track and treat intracranial
pressure, decompress the brain if intracranial pressure is increased, or remove
intracranial hematomas. In the first few days after the injury, maintaining
adequate brain perfusion and oxygenation and preventing complications of altered
sensorium are important.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [52]:
user_input = "I have just broken my leg on a hiking trip and do not have access to a doctor. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 1494 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  117720.87 ms /  1494 tokens (   78.80 ms per token,    12.69 tokens per second)
llama_perf_context_print:        eval time =   26687.66 ms /   127 runs   (  210.14 ms per token,     4.76 tokens per second)
llama_perf_context_print:       total time =  144483.77 ms /  1621 tokens
llama_perf_context_print:    graphs reused =        122


###Answer First, it is important to immobilize the injured leg using a splint or
a cast to prevent further injury and facilitate healing. The joints proximal and
distal to the injury should also be immobilized. If severe swelling is likely,
the cast (and all padding) may need to be cut open from end to end medially and
laterally (bivalved).  Keep the cast dry and never put any object inside it.
Inspect the cast's edges and skin around it every day and apply lotion to any
red or sore areas. Pad any rough edges with


### Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [53]:
user_input = "I am the manager of a critical care unit and I need a detailed protocal for managing and preventing a sepsis case in my unit? Please provide step-by-step instructions for managing sepsis in a critical care unti. Tone shoould be professional in bullet points"
print(generate_rag_response(user_input, max_tokens=1024, top_p=0.9))


Llama.generate: 145 prefix-match hit, remaining 1799 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  143596.35 ms /  1799 tokens (   79.82 ms per token,    12.53 tokens per second)
llama_perf_context_print:        eval time =  108303.84 ms /   502 runs   (  215.74 ms per token,     4.64 tokens per second)
llama_perf_context_print:       total time =  252384.00 ms /  2301 tokens
llama_perf_context_print:    graphs reused =        485


- Identify and suspect sepsis based on clinical presentation:   * Fever or
hypothermia   * Tachycardia or bradycardia   * Tachypnea or respiratory distress
* Altered mental status   * WBC count > 12,000 cells/μL or < 4,000 cells/μL or >
10% immature forms  - Obtain cultures and initiate broad-spectrum antibiotics:
* Blood, body fluids, and wound sites   * Start empiric therapy immediately
after suspecting sepsis   * Antibiotic selection based on suspected source,
clinical setting, and previous culture results  - Administer fluids for
resuscitation:   * Aggressive fluid resuscitation to maintain adequate tissue
perfusion   * Monitor fluid balance and adjust as needed  - Monitor vital signs
and laboratory values:   * Continuous monitoring of vital signs (BP, pulse,
respiration rate, temperature)   * Daily blood tests for electrolytes, CBC, and
coagulation profiles   * Point-of-care testing for quick results  - Provide
supportive care:   * Maintain adequate nutrition   * Prevent infection

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [54]:
user_input = "I am a doctor in a new clinic and I have a patient that may have appendicitis. What are the  common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input, max_tokens=1024))



Llama.generate: 145 prefix-match hit, remaining 1353 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  105749.89 ms /  1353 tokens (   78.16 ms per token,    12.79 tokens per second)
llama_perf_context_print:        eval time =   12089.14 ms /    58 runs   (  208.43 ms per token,     4.80 tokens per second)
llama_perf_context_print:       total time =  117869.85 ms /  1411 tokens
llama_perf_context_print:    graphs reused =         55


###Answer The common symptoms for appendicitis include abdominal pain, anorexia,
and abdominal tenderness. Appendicitis cannot be cured via medicine and requires
surgical removal, specifically an open or laparoscopic appendectomy.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [55]:
user_input = "I am a doctor and my patient has sudden patchy hair loss seen as localized bald spots. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input, max_tokens=1024, temperature=0.1))

Llama.generate: 145 prefix-match hit, remaining 1621 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  130152.93 ms /  1621 tokens (   80.29 ms per token,    12.45 tokens per second)
llama_perf_context_print:        eval time =   25444.40 ms /   120 runs   (  212.04 ms per token,     4.72 tokens per second)
llama_perf_context_print:       total time =  155668.66 ms /  1741 tokens
llama_perf_context_print:    graphs reused =        116


###Answer The effective treatments for sudden patchy hair loss, also known as
alopecia areata, include topical, intralesional, or systemic corticosteroids,
topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or
squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause
behind this condition is believed to be an autoimmune disorder affecting
genetically susceptible people exposed to unclear environmental triggers.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [56]:
user_input = "I have a family member that sustained a physical injury to brain tissue. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input, max_tokens=1024, temperature=0.1))

Llama.generate: 145 prefix-match hit, remaining 1113 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =   88301.67 ms /  1113 tokens (   79.34 ms per token,    12.60 tokens per second)
llama_perf_context_print:        eval time =   32135.11 ms /   156 runs   (  205.99 ms per token,     4.85 tokens per second)
llama_perf_context_print:       total time =  120536.83 ms /  1269 tokens
llama_perf_context_print:    graphs reused =        150


Your answer: The recommended treatments for a person who has sustained a
physical injury to brain tissue, resulting in temporary or permanent impairment
of brain function, include ensuring a reliable airway, maintaining adequate
ventilation, oxygenation, and blood pressure, surgery if needed to place
monitors to track and treat intracranial pressure, decompress the brain if
intracranial pressure is increased, or remove intracranial hematomas. In the
first few days after the injury, maintaining adequate brain perfusion and
oxygenation and preventing complications of altered sensorium are important.
Subsequently, many patients require rehabilitation. Supportive care should
include preventing systemic complications due to immobilization, providing good
nutrition, and preventing pressure ulcers.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [57]:
user_input = "I have just broken my leg on a hiking trip and do not have access to a doctor. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input, max_tokens=1024, temperature=0.1))

Llama.generate: 145 prefix-match hit, remaining 1494 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  117917.76 ms /  1494 tokens (   78.93 ms per token,    12.67 tokens per second)
llama_perf_context_print:        eval time =  117624.74 ms /   550 runs   (  213.86 ms per token,     4.68 tokens per second)
llama_perf_context_print:       total time =  236114.36 ms /  2044 tokens
llama_perf_context_print:    graphs reused =        532


###Answer First, it is important to immobilize the injured leg using a splint or
a cast to prevent further injury and facilitate healing. The joints proximal and
distal to the injury should also be immobilized. If severe swelling is likely,
the cast (and all padding) may need to be cut open from end to end medially and
laterally (bivalved).  Keep the cast dry and never put any object inside it.
Inspect the cast's edges and skin around it every day and apply lotion to any
red or sore areas. Pad any rough edges with soft adhesive tape, cloth, or other
soft material to prevent the cast's edges from causing discomfort or injury.  If
the fracture is not severe, supportive measures such as analgesics and local
wound care may be sufficient. However, if the fracture is severe, delayed
surgery may be necessary.  In the field, frostbitten extremities should be
rewarmed rapidly by immersing them in water that is tolerably warm to the touch
(40 to 42° C). Once in the hospital, core temperature sho

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [58]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [59]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [60]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [61]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [62]:
user_input = "I am the manager of a critical care unit and I need a detailed protocal for managing and preventing a sepsis case in my unit? Please provide step-by-step instructions for managing sepsis in a critical care unti. Tone shoould be professional in bullet points"
generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0,top_p=0.95,top_k=50)

Llama.generate: 1 prefix-match hit, remaining 1958 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  153797.40 ms /  1958 tokens (   78.55 ms per token,    12.73 tokens per second)
llama_perf_context_print:        eval time =  102447.22 ms /   473 runs   (  216.59 ms per token,     4.62 tokens per second)
llama_perf_context_print:       total time =  256695.25 ms /  2431 tokens
llama_perf_context_print:    graphs reused =        458
Llama.generate: 7 prefix-match hit, remaining 2551 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  206049.38 ms /  2551 tokens (   80.77 ms per token,    12.38 tokens per second)
llama_perf_context_print:        eval time =   35883.32 ms /   162 runs   (  221.50 ms per token,     4.51 tokens per second)
llama_perf_context_print:       total time =  242035.33 ms /  2713 tokens
llama_perf_context_print:   

(' Steps to evaluate the answer:\n1. Identify if the answer is derived only from the information presented in the context.\n2. Check if each step in the answer is explicitly stated or can be inferred from the context.\n\nThe answer adheres to the metric as each step is either explicitly stated or can be inferred from the context. The context provides detailed information about managing sepsis in a critical care unit, including symptoms, laboratory findings, and treatment options. The answer uses this information to provide a step-by-step protocol for managing sepsis.\n\nExtent to which the metric is followed: 5 - The metric is followed completely.\n\nRating: 5 (The answer is derived completely from the context and follows the metric perfectly)',
 ' Steps to evaluate the context as per the relevance metric:\n1. Identify the main aspects of the question: managing and preventing a sepsis case in a critical care unit, with a focus on step-by-step instructions.\n2. Review the context to det

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [63]:
user_input = "I am a doctor in a new clinic and I have a patient that may have appendicitis. What are the  common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0,top_p=0.95,top_k=50)

Llama.generate: 7 prefix-match hit, remaining 1506 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  117226.21 ms /  1506 tokens (   77.84 ms per token,    12.85 tokens per second)
llama_perf_context_print:        eval time =   12096.52 ms /    58 runs   (  208.56 ms per token,     4.79 tokens per second)
llama_perf_context_print:       total time =  129354.12 ms /  1564 tokens
llama_perf_context_print:    graphs reused =         55
Llama.generate: 7 prefix-match hit, remaining 1690 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  134134.52 ms /  1690 tokens (   79.37 ms per token,    12.60 tokens per second)
llama_perf_context_print:        eval time =   33171.42 ms /   157 runs   (  211.28 ms per token,     4.73 tokens per second)
llama_perf_context_print:       total time =  167403.67 ms /  1847 tokens
llama_perf_context_print:   

(' Steps to evaluate the answer:\n1. Identify the information in the context related to the question.\n2. Determine if the answer is derived only from the information in the context.\n3. Evaluate the extent to which the metric is followed.\n\nExplanation:\nThe question asks about the common symptoms of appendicitis and if it can be cured via medicine. The context provides information about the symptoms of appendicitis and the treatment for it, which is surgical removal. The AI generated answer states the same information, which is derived directly from the context. Therefore, the metric is followed completely.\n\nRating:\nBased on the evaluation criteria, the answer would receive a score of 5 for following the metric completely.',
 ' Steps to evaluate the context as per the relevance metric:\n1. Identify the main aspects of the question: common symptoms for appendicitis, if it can be cured via medicine, and the surgical procedure for treatment.\n2. Determine if the context contains all

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [64]:
user_input = "I am a doctor and my patient has sudden patchy hair loss seen as localized bald spots. What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0.1,top_p=0.95,top_k=50)

Llama.generate: 7 prefix-match hit, remaining 1774 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  138699.82 ms /  1774 tokens (   78.18 ms per token,    12.79 tokens per second)
llama_perf_context_print:        eval time =   32591.90 ms /   153 runs   (  213.02 ms per token,     4.69 tokens per second)
llama_perf_context_print:       total time =  171388.83 ms /  1927 tokens
llama_perf_context_print:    graphs reused =        147
Llama.generate: 7 prefix-match hit, remaining 2053 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  163434.29 ms /  2053 tokens (   79.61 ms per token,    12.56 tokens per second)
llama_perf_context_print:        eval time =   45578.38 ms /   212 runs   (  214.99 ms per token,     4.65 tokens per second)
llama_perf_context_print:       total time =  209160.78 ms /  2265 tokens
llama_perf_context_print:   

(' Steps to evaluate the answer:\n1. Identify the question and context.\n2. Understand the question, which is asking for effective treatments and possible causes for sudden patchy hair loss, commonly known as alopecia areata.\n3. Identify the information in the context related to alopecia areata and its treatments.\n4. Check if the answer is derived only from the information presented in the context.\n\nThe answer adheres to the metric as it mentions the possible causes (autoimmune disorders and unclear environmental triggers) and effective treatments (topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy, or PUVA) for alopecia areata, all of which are present in the context.\n\nRating:\n5 - The metric is followed completely.\n\nThe answer is a complete and accurate reflection of the information provided in the context.',
 ' Steps to evaluate the context as per the relevance metric:\n1. Identify the main aspects of the question

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [65]:
user_input = "I have a family member that sustained a physical injury to brain tissue. What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0.1,top_p=0.95,top_k=50)

Llama.generate: 7 prefix-match hit, remaining 1266 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =   97462.93 ms /  1266 tokens (   76.98 ms per token,    12.99 tokens per second)
llama_perf_context_print:        eval time =   38734.69 ms /   185 runs   (  209.38 ms per token,     4.78 tokens per second)
llama_perf_context_print:       total time =  136322.58 ms /  1451 tokens
llama_perf_context_print:    graphs reused =        178
Llama.generate: 7 prefix-match hit, remaining 1577 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  123163.92 ms /  1577 tokens (   78.10 ms per token,    12.80 tokens per second)
llama_perf_context_print:        eval time =   54128.88 ms /   256 runs   (  211.44 ms per token,     4.73 tokens per second)
llama_perf_context_print:       total time =  177484.76 ms /  1833 tokens
llama_perf_context_print:   

(' Steps to evaluate the answer:\n1. Identify the key information in the context related to the recommended treatments for a person with a physical injury to brain tissue.\n2. Compare the information in the context to the answer to ensure that the answer is derived only from the context and not from any external sources.\n3. Evaluate the extent to which the answer adheres to the metric.\n\nExplanation:\nThe answer mentions the recommended treatments for a person with a physical injury to brain tissue based on the context. The context provides information about the initial treatment, which includes ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and the need for surgery in patients with more severe injuries. The answer also mentions the importance of preventing complications in the first few days and the requirement for rehabilitation subsequently. The answer also includes supportive care measures mentioned in a different chapter of the con

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [66]:
user_input = "I have just broken my leg on a hiking trip and do not have access to a doctor. What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0.1,top_p=0.95,top_k=50)


Llama.generate: 7 prefix-match hit, remaining 1647 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  131620.84 ms /  1647 tokens (   79.92 ms per token,    12.51 tokens per second)
llama_perf_context_print:        eval time =   52331.65 ms /   248 runs   (  211.01 ms per token,     4.74 tokens per second)
llama_perf_context_print:       total time =  184134.15 ms /  1895 tokens
llama_perf_context_print:    graphs reused =        239
Llama.generate: 7 prefix-match hit, remaining 2021 prompt tokens to eval
llama_perf_context_print:        load time =     939.83 ms
llama_perf_context_print: prompt eval time =  159613.76 ms /  2021 tokens (   78.98 ms per token,    12.66 tokens per second)
llama_perf_context_print:        eval time =   45443.69 ms /   209 runs   (  217.43 ms per token,     4.60 tokens per second)
llama_perf_context_print:       total time =  205203.81 ms /  2230 tokens
llama_perf_context_print:   

(' Steps to evaluate the answer:\n1. Identify the information in the context related to the question.\n2. Check if the answer is derived only from the information in the context.\n3. Compare each step in the answer to the corresponding information in the context.\n\nEvaluation:\nThe answer adheres to the metric to a good extent. The answer mentions all the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, which are directly derived from the context. However, the answer also includes some additional information about frostbitten extremities and rewarming, which is not directly related to the question or context. Therefore, the answer is not following the metric completely.\n\nRating:\nBased on the evaluation, I would rate the answer as a 3 on the given evaluation criteria. The answer follows the metric to a good extent, but there is some additional information that is not directly related to the question or context.',
 ' Steps to ev

## Actionable Insights and Business Recommendations

1.   If this project is going to move forward the appropriate invesment in upgraded hareware must be made.
2.   With the appropaite hadeware, this could be a very valuable system for doctors, thier patients and anyone in the medical field. It would provide advice and guidance to doctors and help patients and their care takers more information about conditions and alternative treatments.
3. The RAG data would need to be keep current with addiitonal information.
4. Figure out how to get the result to print out in an easier to read format.



<font size=6 color='blue'>Power Ahead</font>
___

In [67]:
end_time = time.time()
total_time = end_time - my_start_time
total_time/60

95.34605074326197

In [3]:
# In a new cell
!jupyter nbconvert --clear-output Craig_Murray_Project.ipynb


[NbConvertApp] Converting notebook Craig_Murray_Project.ipynb to notebook
[NbConvertApp] Writing 164995 bytes to Craig_Murray_Project.ipynb


In [1]:
'''
import nbformat

with open("Craig_Murray_Project.ipynb", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# Write cleaned notebook
with open("your_notebook_clean.ipynb", "w") as f:
    nbformat.write(nb, f)
'''

'\nimport nbformat\n\nwith open("Craig_Murray_Project.ipynb", encoding="utf-8") as f:\n    nb = nbformat.read(f, as_version=4)\n\n# Write cleaned notebook\nwith open("your_notebook_clean.ipynb", "w") as f:\n    nbformat.write(nb, f)\n'